# OLR Quicklook (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-06-24  
**Last Modified:** 2026-06-24  
**Status:** In Progress  
**Keywords:** AOS, open loop reproduction, wavefront, Zernikes, DOF  

## Description

Quicklook diagnostics for the Open Loop Reproduction (OLR) pipeline outputs in
`rubin-work/olr`. Reads the two per-night parquet products and compares the
open-loop reproduction against the closed-loop (measured) wavefront.

Key functionality:
1. Read the per-night nightly AOS table and the OLR parquet for one `day_obs`.
2. Compare open-loop (reproduced) vs closed-loop (measured) Zernikes and the
   wavefront RMS the AOS loop removed, with PSF FWHM overlaid.
3. Show the applied AOS correction (the trim added back to make the OLR), split
   by component: camera & M2 piston, decenter, tip/tilt, and the M1M3 / M2
   bending modes.

**Output:** Inline diagnostic plots (no files written).

**Based on:** `rubin-work/olr` pipeline; Craig Lage's Open_Loop_Reproduction
notebook; canonical `nightly_report_ts_version.ipynb` (DM-54406).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-24 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs = 20260415                 # night to inspect
base    = f"output/{day_obs}"      # pipeline output dir (run from rubin-work/olr)

# Which wavefront to compare: "deviation" (controlled quantity) or "opd"
# (the full optical path difference, used for the measured-intrinsic / MIW work).
field = "deviation"

# Zernike panels to show (label, Noll index).
zernike_panels = [("Z4 defocus", 4), ("Z5 astig", 5), ("Z6 astig", 6),
                  ("Z7 coma", 7), ("Z8 coma", 8), ("Z11 spherical", 11)]

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
CORNER_NAMES = ["R00", "R04", "R40", "R44"]   # 4 corner WFS sensors
NOLL = np.arange(4, 27)                        # 23 terms; array index i <-> Noll i+4

# --- 50-DOF layout (OFC aggregatedDoF order) ---
# 0-4   M2 hexapod  : dz, dx, dy, rx, ry
# 5-9   Cam hexapod : dz, dx, dy, rx, ry
# 10-29 M1M3 bending modes (20)
# 30-49 M2   bending modes (20)
DZ_COLS      = [0, 5]                  # piston            (microns)
DXDY_COLS    = [1, 2, 6, 7]            # decenter          (microns)
TIPTILT_COLS = [3, 4, 8, 9]            # tip/tilt          (arcsec)
M1M3_COLS    = list(range(10, 30))
M2BEND_COLS  = list(range(30, 50))


def stack(df, prefix):
    """Per-corner column set {prefix}_R00.. -> (n_visits, 4, 23) array."""
    return np.stack([np.vstack(df[f"{prefix}_{c}"].values) for c in CORNER_NAMES], axis=1)


def term(arr, noll):
    """Pick a single Noll term from a (..., 23) array."""
    return arr[..., noll - 4]


def plot_dof_lines(ax, seq, dof, cols, labels, ylabel, title):
    """Line plot of selected DOF columns vs seq."""
    for col, lab in zip(cols, labels):
        ax.plot(seq, dof[:, col], ".-", ms=4, lw=0.8, label=lab)
    ax.axhline(0, lw=0.6, color="k", alpha=0.4)
    ax.set_xlabel("seq")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)


def plot_bending_heatmap(seq, dof, cols, title, mode_label):
    """Heatmap of a block of bending modes (mode index vs seq)."""
    sub = dof[:, cols].T
    vmax = np.nanpercentile(np.abs(sub), 98) or 1.0
    fig, ax = plt.subplots(figsize=(11, 5))
    im = ax.imshow(sub, aspect="auto", origin="lower", cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax,
                   extent=[seq.min(), seq.max(), 0.5, len(cols) + 0.5])
    ax.set_xlabel("seq")
    ax.set_ylabel(mode_label)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="amplitude [microns]")
    fig.tight_layout()
    return fig

<a id='data'></a>
## Data Access

In [ ]:
olr_df = pd.read_parquet(f"{base}/olr.parquet")
night  = pd.read_parquet(f"{base}/nightly_aos_table.parquet")

print(f"olr.parquet            : {olr_df.shape[0]} rows, {olr_df.shape[1]} cols")
print(f"nightly_aos_table      : {night.shape[0]} rows, {night.shape[1]} cols")
print("bands:", sorted(olr_df["band"].dropna().unique().tolist()))

<a id='analysis'></a>
## Analysis

In [ ]:
olr_df = olr_df.sort_values("seq").reset_index(drop=True)
seq = olr_df["seq"].values

# Mean over the 4 corner sensors -> (n_visits, 23).
open_wf   = stack(olr_df, f"olr_{field}").mean(axis=1)    # open-loop reproduction
closed_wf = stack(olr_df, f"meas_{field}").mean(axis=1)   # measured (closed-loop)

# Wavefront RMS in quadrature over Noll 4..26.
rms_open   = np.sqrt(np.nansum(open_wf**2,   axis=1))
rms_closed = np.sqrt(np.nansum(closed_wf**2, axis=1))

# Full 50-DOF state (the correction; trim = dof_state[active 22]).
dof = np.vstack(olr_df["dof_state"].values)

# PSF FWHM from the nightly table, aligned to the OLR seqs.
fwhm = night.set_index("seq")["psf_fwhm"].reindex(seq).values

print(f"field = {field!r};  {len(seq)} visits;  dof state {dof.shape}")

<a id='results'></a>
## Results & Plots

### Open- vs closed-loop Zernikes across the night

In [ ]:
fig, axes = plt.subplots(len(zernike_panels), 1, figsize=(11, 12), sharex=True)
for ax, (label, n) in zip(axes, zernike_panels):
    ax.plot(seq, term(closed_wf, n), ".", ms=4, color="tab:blue",   label="closed-loop (measured)")
    ax.plot(seq, term(open_wf,   n), ".", ms=4, color="tab:orange", label="open-loop reproduction")
    ax.axhline(0, lw=0.6, color="k", alpha=0.4)
    ax.set_ylabel(f"{label}\n[microns]")
    ax.grid(alpha=0.3)
axes[0].legend(ncol=2, fontsize=9, loc="upper right")
axes[-1].set_xlabel("seq")
fig.suptitle(f"OLR vs measured Zernikes ({field}) — {day_obs}", y=0.995)
fig.tight_layout()

### Wavefront RMS the loop removed (with PSF FWHM)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(seq, rms_open,   ".", ms=5, color="tab:orange", label="open-loop RMS WF")
ax.plot(seq, rms_closed, ".", ms=5, color="tab:blue",   label="closed-loop RMS WF")
ax.set_xlabel("seq")
ax.set_ylabel("WF RMS  [microns]")
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(seq, fwhm, "-", lw=1, color="gray", alpha=0.7)
ax2.set_ylabel("PSF FWHM [arcsec]", color="gray")
ax.legend(loc="upper left")
ax.set_title(f"Wavefront RMS: closed vs open loop ({field}) — {day_obs}")
fig.tight_layout()

### Applied AOS correction (trim), split by component

The trim added back to make the OLR, from `dof_state`. Hexapod piston/decenter
are in microns, tip/tilt in arcsec; bending modes (microns) are shown as
heatmaps so the small modes are not masked by the large hexapod piston.

In [ ]:
# Camera & M2 piston (dz)
fig, ax = plt.subplots(figsize=(11, 4))
plot_dof_lines(ax, seq, dof, DZ_COLS, ["M2 dz", "Cam dz"],
               "piston [microns]", f"Camera & M2 piston (dz) — {day_obs}")
fig.tight_layout()

In [ ]:
# Camera & M2 decenter (dx, dy)
fig, ax = plt.subplots(figsize=(11, 4))
plot_dof_lines(ax, seq, dof, DXDY_COLS, ["M2 dx", "M2 dy", "Cam dx", "Cam dy"],
               "decenter [microns]", f"Camera & M2 decenter (dx, dy) — {day_obs}")
fig.tight_layout()

In [ ]:
# Camera & M2 tip/tilt (rx, ry)
fig, ax = plt.subplots(figsize=(11, 4))
plot_dof_lines(ax, seq, dof, TIPTILT_COLS, ["M2 rx", "M2 ry", "Cam rx", "Cam ry"],
               "tip/tilt [arcsec]", f"Camera & M2 tip/tilt (rx, ry) — {day_obs}")
fig.tight_layout()

In [ ]:
# M1M3 bending modes
plot_bending_heatmap(seq, dof, M1M3_COLS, f"M1M3 bending modes — {day_obs}", "M1M3 mode #");

In [ ]:
# M2 bending modes
plot_bending_heatmap(seq, dof, M2BEND_COLS, f"M2 bending modes — {day_obs}", "M2 mode #");